# Train scVI model
Reference GSE254789

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

In [ ]:
from pathlib import Path
import scanpy as sc
from scipy.sparse import issparse
import scvi
import numpy as np
from collections import Counter
import anndata as ad
import matplotlib.pyplot as plt
from lightning.pytorch import seed_everything
import random
import torch
import sys
import session_info
import os

In [ ]:
random.seed(0)
seed_everything(0)

scvi.settings.seed = 0
scvi.settings.num_workers = 32

## Set paths, import data

In [ ]:
# Directories
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import BASE_DIR
print('BASE_DIR:', BASE_DIR)

# input ref data
input_dir = BASE_DIR / "data" / "scrna-seq" / "h5ad" / "02_filtered"

# output locations
h5ad_dir = BASE_DIR / "data" / "scrna-seq" / "h5ad" / "03_scvi"
model_dir = BASE_DIR / "scvi" / "scvi_refA"
h5ad_dir.mkdir(parents=True, exist_ok=True)

# File
h5ad_out = h5ad_dir / "GSE254789-scvi.h5ad"

In [ ]:
adata_path = input_dir / "GSE254789-clean.h5ad"

In [ ]:
adata = sc.read_h5ad(adata_path)
adata=adata[adata.obs['predicted_doublets'] == False].copy()
adata

In [ ]:
adata.obs.sample_id.value_counts()

## Prepare adata

In [ ]:
adata.X[:5, :5].toarray()

In [ ]:
# Save counts layer for scVI and downstream processing
adata.X = adata.layers["counts"].copy()

print('adata.X is sparse:', issparse(adata.X))
print('adata.X has only whole numbers:', np.all(adata.X.data == np.round(adata.X.data)))  # True if all values are whole numbers

## scVI

#### Train model

In [ ]:
# Set up AnnData for SCVI
scvi.model.SCVI.setup_anndata(
    adata, 
    layer="counts",  # Use raw count data
    batch_key = 'sample_id'
)

# Initialize model
model = scvi.model.SCVI(
    adata
)

# Train SCVI model
model.train(
    early_stopping=True,
    accelerator = "gpu",
    enable_progress_bar=True
)

model.save(model_dir, prefix='20250612B_', overwrite=True)

# add scVI latent dimensions to adata.obsm
adata.obsm['X_scVI'] = model.get_latent_representation(adata).astype(np.float32)
adata.obsm['X_scVI'].shape

# save
adata.write_h5ad(h5ad_out, compression='gzip')

In [ ]:
plt.plot(model.history["elbo_train"], label="Train ELBO Loss")
plt.plot(model.history["elbo_validation"], label="Validation ELBO Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("SCVI Training Loss Curve")
plt.show()

In [ ]:
adata.obsm['X_scVI'].shape